# Titanic 생존 예측: Threshold와 Class Weight 실험

## 학습 목표

기존 로지스틱 회귀 baseline의 feature는 `Sex`, `Pclass`, `FamilySize`, `AgeGroup`으로 유지한다.

이번 노트북에서는 다음 두 방법이 생존자 탐지 성능에 미치는 영향을 비교한다.

1. `predict_proba()`로 구한 생존확률의 판정 기준(threshold)을 `0.5`보다 낮추기
2. `LogisticRegression(class_weight="balanced")`로 소수 클래스인 생존자의 오분류에 더 큰 가중치 주기

Accuracy만 보지 않고 생존 클래스의 Recall, Precision, F1과 혼동행렬을 함께 비교한다.

## 1. 라이브러리

- `pandas`, `numpy`: 데이터 처리와 feature 생성
- `ColumnTransformer`, `Pipeline`: 숫자형·범주형 전처리와 모델을 하나의 흐름으로 연결
- `cross_val_predict`: 각 승객에 대한 out-of-fold 예측 생성
- `sklearn.metrics`: 혼동행렬과 분류 성능 평가

In [3]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report
)
from sklearn.model_selection import (
    StratifiedKFold,
    cross_val_predict,
    cross_val_score
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix

print("Ready")

Ready


## 2. 데이터 불러오기

Kaggle Titanic의 학습 데이터, 테스트 데이터, 제출 예시 파일을 불러온다. 학습 데이터에는 `Survived`가 있지만 실제 제출용 테스트 데이터에는 정답 열이 없다.

결측치 비율을 확인하면 `Age`, `Cabin`, `Embarked`에 결측치가 있다. 이번 모델에서는 `Age`를 구간화한 `AgeGroup`을 사용하며, 이때 생긴 결측치는 Pipeline 내부에서 처리한다.

In [4]:
train = pd.read_csv("/kaggle/input/competitions/titanic/train.csv")
test = pd.read_csv("/kaggle/input/competitions/titanic/test.csv")
gender_submission = pd.read_csv(
    "/kaggle/input/competitions/titanic/gender_submission.csv"
)

# 데이터에 어떤 열이 있는지 확인
print(train.columns)

# 열별 결측치 비율(%): 여러 표현식 중 마지막 결과가 셀의 출력으로 표시된다.
train.isna().mean().mul(100)

Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked'],
      dtype='object')


PassengerId     0.000000
Survived        0.000000
Pclass          0.000000
Name            0.000000
Sex             0.000000
Age            19.865320
SibSp           0.000000
Parch           0.000000
Ticket          0.000000
Fare            0.000000
Cabin          77.104377
Embarked        0.224467
dtype: float64

## 3. Baseline feature 생성

지난 실험에서 선택한 네 feature를 그대로 사용한다.

- `Sex`: 성별
- `Pclass`: 객실 등급. 숫자로 저장되어 있지만 등급별 효과를 독립적으로 보기 위해 범주형으로 처리한다.
- `FamilySize = SibSp + Parch + 1`: 함께 탑승한 형제·배우자와 부모·자녀 수에 자기 자신을 더한 가족 규모
- `AgeGroup`: 나이를 `[0, 6)`, `[6, 20)`, `[20, 40)`, `[40, 60)`, `[60, ∞)`로 구간화

`Age`가 결측이면 `pd.cut()`의 결과인 `AgeGroup`도 `NaN`이 된다. 이 값은 뒤의 범주형 Pipeline에서 최빈 범주로 대체된다.

In [5]:
def create_features(data):
    # 원본을 바꾸지 않고 FamilySize와 AgeGroup을 추가한다.
    result = data.copy()

    # 동반 가족 수에 승객 본인을 더한다.
    result["FamilySize"] = result["SibSp"] + result["Parch"] + 1

    # right=False이므로 왼쪽 경계는 포함하고 오른쪽 경계는 포함하지 않는다.
    result["AgeGroup"] = pd.cut(
        result["Age"],
        bins=[0, 6, 20, 40, 60, np.inf],
        labels=["Preschool", "Teen", "20-30", "40-50", "60+"],
        right=False,
    )
    return result


train_fe = create_features(train)
test_fe = create_features(test)

features = ["Sex", "Pclass", "FamilySize", "AgeGroup"]

X = train_fe[features]
y = train_fe["Survived"]

X

,Sex,Pclass,FamilySize,AgeGroup
0,male,3,2,20-30
1,female,1,2,20-30
2,female,3,1,20-30
3,female,1,2,20-30
4,male,3,1,20-30
...,...,...,...,...
886,male,2,1,20-30
887,female,1,1,Teen
888,female,3,4,NaN
889,male,1,1,20-30


## 4. 숫자형·범주형 전처리 Pipeline

### 숫자형: `FamilySize`

1. 결측치가 있으면 중앙값으로 대체한다. 현재 train에 결측치가 없더라도 이후 다른 데이터에 생길 가능성까지 Pipeline이 처리하도록 한다.
2. `StandardScaler`로 평균 0, 표준편차 1에 가깝게 표준화한다.

### 범주형: `Sex`, `Pclass`, `AgeGroup`

1. 결측치를 학습 fold에서 가장 자주 나타난 값으로 대체한다.
2. `OneHotEncoder`로 각 범주를 별도의 0/1 열로 변환한다.
3. `handle_unknown="ignore"`는 검증 fold나 test에 학습 fold에서 보지 못한 범주가 나타났을 때 오류가 나는 것을 막는다.

중요한 점은 `SimpleImputer`가 원본 `train_fe`를 수정하지 않는다는 것이다. 따라서 원본 표에는 `AgeGroup=NaN`이 그대로 보이지만, 모델에 들어갈 때만 Pipeline 내부에서 대체된다. 교차검증에서는 각 fold의 학습 데이터로 최빈값을 따로 계산하므로 데이터 누수도 방지한다.

In [6]:
numerical_features = ["FamilySize"]
categorical_features = ["Sex", "Pclass", "AgeGroup"]

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

# 각 열 그룹에 알맞은 전처리를 적용한 뒤 결과를 하나의 행렬로 합친다.
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

## 5. Logistic Regression Pipeline

전처리와 로지스틱 회귀를 하나의 Pipeline으로 묶는다.

`원본 feature → 결측치 처리 → 표준화/One-Hot Encoding → Logistic Regression`

이렇게 묶으면 교차검증의 각 fold 안에서 전처리도 다시 학습되므로 검증 데이터의 정보가 학습 단계에 미리 들어가는 것을 막을 수 있다. `max_iter=1000`은 최적화가 충분히 수렴할 수 있도록 최대 반복 횟수를 늘린 것이다.

In [7]:
baseline_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(max_iter=1000)),
    ]
)

## 6. 5-fold 교차검증과 OOF 예측

`StratifiedKFold`는 각 fold에 생존자와 사망자의 비율이 전체 데이터와 비슷하게 유지되도록 나눈다. `shuffle=True`로 나누기 전에 행을 섞고, `random_state=42`로 같은 분할을 재현한다.

`cross_val_predict(..., method="predict")`가 만드는 OOF(out-of-fold) 예측에서는 각 승객이 자신을 학습에 사용하지 않은 모델로부터 정확히 한 번 예측을 받는다. 이는 전체 데이터로 학습하고 같은 데이터에 다시 예측하는 것과 다르다.

In [18]:
cv = StratifiedKFold(  
    n_splits = 5, 
    shuffle = True, 
    random_state = 42
)


oof_pred = cross_val_predict(
    estimator = baseline_pipeline, 
    X = X, 
    y = y, 
    cv = cv , 
    method = "predict", 
    n_jobs = -1 
)

print(oof_pred[:20])

comparison = X.copy()
comparison["actual"] = y 
comparison["oof_pred"] = oof_pred
comparison["is_correct"] = comparison["actual"] == comparison["oof_pred"]

# X의 행 index에 맞춰 원본 age 결측 여부 가져오기 
comparison["Age_was_missing"] = train.loc[X.index, "Age"].isna()
comparison.head(20)

comparison.groupby("Age_was_missing").agg(
    passenger_count=("actual", "size"),
    error_count=("is_correct", lambda x: (~x).sum()),
    error_rate=("is_correct", lambda x: (~x).mean()),
    accuracy=("is_correct", "mean")
)

[0 1 1 1 0 0 0 0 1 1 1 1 0 0 1 1 0 0 1 1]


,passenger_count,error_count,error_rate,accuracy
Age_was_missing,,,,
False,714,135,0.189076,0.810924
True,177,32,0.180791,0.819209


In [9]:
# confusion_matrix의 배열 순서는 [[TN, FP], [FN, TP]]이다.
cm = confusion_matrix(y, oof_pred)
cm

array([[482,  67],
       [100, 242]])

### Baseline 혼동행렬 해석

| 실제값 / 예측값 | 사망 예측(0) | 생존 예측(1) |
|---|---:|---:|
| 실제 사망(0) | TN = 482 | FP = 67 |
| 실제 생존(1) | FN = 100 | TP = 242 |

- Accuracy: `(482 + 242) / 891 = 0.813`
- 생존 Recall: `242 / (242 + 100) = 0.708`
- 생존 Precision: `242 / (242 + 67) = 0.783`
- F1: `0.743`

실제 생존자 342명 중 242명을 생존자로 찾았고, 100명은 사망으로 잘못 예측했다.

## 7. `predict_proba()`와 기본 threshold 확인

`predict()`는 최종 클래스인 0 또는 1을 반환하지만, `predict_proba()`는 각 클래스에 속할 추정확률을 반환한다. Titanic에서 목표값은 `0=사망`, `1=생존`이며 scikit-learn은 클래스 순서 `[0, 1]`에 맞춰 확률 열을 반환한다.

- `oof_proba[:, 0]`: 사망(0) 확률
- `oof_proba[:, 1]`: 생존(1) 확률

확률은 연속값이므로 혼동행렬에 바로 넣을 수 없다. 먼저 threshold를 적용해 0/1 예측으로 바꿔야 한다.

In [10]:
oof_proba = cross_val_predict(
    estimator=baseline_pipeline,
    X=X,
    y=y,
    cv=cv,
    method="predict_proba",
    n_jobs=-1,
)

oof_proba[:5]

array([[0.91840871, 0.08159129],
       [0.08339345, 0.91660655],
       [0.34470418, 0.65529582],
       [0.06887721, 0.93112279],
       [0.8886667 , 0.1113333 ]])

In [11]:
# y에서 0은 사망, 1은 생존이므로 두 번째 열이 생존확률이다.
survival_proba = oof_proba[:, 1]

# 비교 결과인 Boolean을 int로 바꾸면 False→0, True→1이 된다.
pred_05 = (survival_proba >= 0.5).astype(int)

In [12]:
print(pred_05[:20])
print(oof_pred[:20])

# predict()와 생존확률에 0.5를 적용한 결과가 모두 같은지 확인한다.
print((pred_05 == oof_pred).mean())

[0 1 1 1 0 0 0 0 1 1 1 1 0 0 1 1 0 0 1 1]
[0 1 1 1 0 0 0 0 1 1 1 1 0 0 1 1 0 0 1 1]
1.0


In [13]:
# 생존/사망의 인원수와 비율을 확인한다.
print(y.value_counts())
print(y.value_counts(normalize=True))
print(confusion_matrix(y, oof_pred))
print((pred_05 == oof_pred).mean())

Survived
0    549
1    342
Name: count, dtype: int64
Survived
0    0.616162
1    0.383838
Name: proportion, dtype: float64
[[482  67]
 [100 242]]
1.0


## 8. Threshold 실험

threshold를 낮추면 생존이라고 판정하기 쉬워진다. 따라서 일반적으로 TP와 Recall은 증가하고 FN은 감소한다. 반면 FP가 증가해 Precision이 낮아질 수 있다. 모델을 다시 학습하는 것은 아니며, 같은 `survival_proba`를 0과 1로 나누는 최종 기준만 바꾼다.

주요 지표는 다음과 같다.

- `Recall = TP / (TP + FN)`: 실제 생존자 중 생존으로 찾아낸 비율
- `Precision = TP / (TP + FP)`: 생존 예측 중 실제 생존자의 비율
- `F1 = 2 × Precision × Recall / (Precision + Recall)`: Precision과 Recall의 조화평균

In [14]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
)

thresholds = [0.5, 0.475, 0.45, 0.4, 0.35, 0.3]
threshold_results = []

for threshold in thresholds:
    # 반복할 때마다 현재 threshold에 맞는 0/1 예측을 새로 만든다.
    threshold_pred = (survival_proba >= threshold).astype(int)

    # ravel()은 2×2 혼동행렬을 TN, FP, FN, TP 순서의 1차원 값으로 푼다.
    tn, fp, fn, tp = confusion_matrix(y, threshold_pred).ravel()

    threshold_results.append(
        {
            "threshold": threshold,
            "TN": tn,
            "FP": fp,
            "FN": fn,
            "TP": tp,
            "accuracy": accuracy_score(y, threshold_pred),
            "recall": recall_score(y, threshold_pred),
            # 생존 예측이 0명인 경우 0으로 나누는 경고 대신 precision=0으로 처리한다.
            "precision": precision_score(
                y, threshold_pred, zero_division=0
            ),
            "f1": f1_score(y, threshold_pred),
            "predicted_survivors": threshold_pred.sum(),
        }
    )

threshold_summary = pd.DataFrame(threshold_results)
threshold_summary

,threshold,TN,FP,FN,TP,accuracy,recall,precision,f1,predicted_survivors
0,0.500,482,67,100,242,0.812570,0.707602,0.783172,0.743472,309
1,0.475,468,81,91,251,0.806958,0.733918,0.756024,0.744807,332
2,0.450,461,88,84,258,0.806958,0.754386,0.745665,0.750000,346
3,0.400,445,104,80,262,0.793490,0.766082,0.715847,0.740113,366
4,0.350,427,122,71,271,0.783389,0.792398,0.689567,0.737415,393
5,0.300,408,141,60,282,0.774411,0.824561,0.666667,0.737255,423


### Threshold 결과 해석

threshold `0.50 → 0.45`의 변화:

- TP: `242 → 258` — 생존자 16명을 추가로 발견
- FN: `100 → 84` — 놓친 생존자 16명 감소
- FP: `67 → 88` — 사망자 21명을 생존으로 추가 오분류
- Recall: `0.708 → 0.754`
- Precision: `0.783 → 0.746`
- Accuracy: `0.813 → 0.807`
- F1: `0.743 → 0.750`

Accuracy의 감소는 작고 Recall은 높아졌으며, 실험한 값 중 F1도 `0.45`에서 가장 높다. 따라서 현재 후보에서는 **threshold 0.45가 Recall과 Precision의 가장 균형적인 절충안**이다.

단, 예측 생존자 346명이 실제 생존자 342명과 비슷하다는 이유로 좋은 threshold인 것은 아니다. 인원수가 비슷해도 구성원이 틀릴 수 있으므로 TP, FP, FN과 평가 지표를 기준으로 판단해야 한다.

In [31]:
# 0.5에서는 사망, 0.45에서는 생존으로 바뀐 승객의 확률 범위는 0.45 <= p < 0.5이다.
pred_045 = (survival_proba >= 0.45).astype(int)

comparison_05_045 = train_fe.copy()
comparison_05_045["survival_proba"] = survival_proba
comparison_05_045["pred_05"] = pred_05
comparison_05_045["pred_045"] = pred_045
comparison_05_045["changed"] = pred_05 != pred_045

changed_passengers = comparison_05_045[
    comparison_05_045["changed"]
].copy()

# 총 37명의 예측이 바뀌며, 이 중 실제 생존자는 16명이고 사망자는 21명이다.
print(changed_passengers.shape)

# 필요하면 아래 주석을 풀어 어떤 승객의 예측이 바뀌었는지 확인한다.
# changed_passengers[
#     features
#     + ["Survived", "survival_proba", "pred_05", "pred_045"]
# ].sort_values("survival_proba")
# changed_passengers["Survived"].value_counts()

(37, 18)


## 9. `class_weight="balanced"` 실험

이번에는 우선 threshold를 0.5로 고정하고 학습 과정 자체를 바꾼다.

`class_weight="balanced"`는 Titanic의 실제 생존율을 50:50이라고 가정하는 옵션이 아니다. 각 클래스의 가중치를 `전체 표본 수 / (클래스 수 × 해당 클래스 표본 수)`로 정해, 표본 수가 적은 클래스의 오류가 학습 손실에 더 크게 반영되도록 한다.

현재 데이터에서는 대략 다음 가중치가 적용된다.

- 사망(0): `891 / (2 × 549) ≈ 0.812`
- 생존(1): `891 / (2 × 342) ≈ 1.303`

따라서 생존자 한 명의 오분류가 사망자 한 명의 오분류보다 크게 반영된다. 생존 Recall은 올라갈 가능성이 있지만 Precision과 Accuracy가 함께 좋아진다는 보장은 없다.

In [29]:
balanced_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced",
            ),
        ),
    ]
)

In [30]:
# predict()는 기본 threshold 0.5를 적용한 최종 클래스를 반환한다.
balanced_oof_pred = cross_val_predict(
    estimator=balanced_pipeline,
    X=X,
    y=y,
    cv=cv,
    method="predict",
    n_jobs=-1,
)

tn, fp, fn, tp = confusion_matrix(y, balanced_oof_pred).ravel()

balanced_result = pd.DataFrame(
    [
        {
            "model": "class_weight = balanced",
            "threshold": 0.5,
            "TN": tn,
            "FP": fp,
            "FN": fn,
            "TP": tp,
            "accuracy": accuracy_score(y, balanced_oof_pred),
            "recall": recall_score(y, balanced_oof_pred),
            "precision": precision_score(
                y, balanced_oof_pred, zero_division=0
            ),
            "f1": f1_score(y, balanced_oof_pred),
            "predicted_survivors": balanced_oof_pred.sum(),
        }
    ]
)

balanced_result

,model,threshold,TN,FP,FN,TP,accuracy,recall,precision,f1,predicted_survivors
0,class_weight = balanced,0.5,443,106,80,262,0.791246,0.766082,0.711957,0.738028,368


`balanced + threshold 0.5`는 baseline 0.5보다 Recall을 `0.708 → 0.766`으로 높였다. 그러나 Accuracy는 `0.813 → 0.791`, Precision은 `0.783 → 0.712`, F1은 `0.743 → 0.738`로 낮아졌다.

또한 Recall이 같은 baseline threshold 0.4와 비교해도 F1이 더 높지 않았다. 따라서 현재 데이터에서는 class weight가 새로운 개선을 만들었다기보다 생존 예측을 늘리는 방향으로 결정 경계를 옮긴 효과에 가깝다.

### Balanced 모델에서도 threshold를 0.45로 낮추기

`class_weight="balanced"`와 threshold 감소는 모두 생존 예측을 늘리는 방향으로 작용한다. 두 조정을 함께 적용했을 때 Recall이 더 높아지는 대신 FP가 얼마나 증가하는지 확인한다.

In [34]:
balanced_oof_proba = cross_val_predict(
    estimator=balanced_pipeline,
    X=X,
    y=y,
    cv=cv,
    method="predict_proba",
    n_jobs=-1,
)

# 두 번째 열은 balanced 모델이 추정한 생존(1) 확률이다.
balanced_pred_045 = (balanced_oof_proba[:, 1] >= 0.45).astype(int)

tn, fp, fn, tp = confusion_matrix(y, balanced_pred_045).ravel()

balanced_045_result = pd.DataFrame(
    [
        {
            "model": "class_weight = balanced",
            "threshold": 0.45,
            "TN": tn,
            "FP": fp,
            "FN": fn,
            "TP": tp,
            "accuracy": accuracy_score(y, balanced_pred_045),
            "recall": recall_score(y, balanced_pred_045),
            "precision": precision_score(
                y, balanced_pred_045, zero_division=0
            ),
            "f1": f1_score(y, balanced_pred_045),
            "predicted_survivors": balanced_pred_045.sum(),
        }
    ]
)

balanced_045_result

,model,threshold,TN,FP,FN,TP,accuracy,recall,precision,f1,predicted_survivors
0,class_weight = balanced,0.45,423,126,69,273,0.781145,0.798246,0.684211,0.736842,399


## 10. 최종 비교

In [35]:
final_comparison = pd.concat(
    [
        threshold_summary[
            threshold_summary["threshold"].isin([0.5, 0.45])
        ].assign(model="baseline"),
        balanced_result,
        balanced_045_result,
    ],
    ignore_index=True,
)

final_comparison[
    [
        "model",
        "threshold",
        "TN",
        "FP",
        "FN",
        "TP",
        "accuracy",
        "recall",
        "precision",
        "f1",
        "predicted_survivors",
    ]
]

,model,threshold,TN,FP,FN,TP,accuracy,recall,precision,f1,predicted_survivors
0,baseline,0.50,482,67,100,242,0.812570,0.707602,0.783172,0.743472,309
1,baseline,0.45,461,88,84,258,0.806958,0.754386,0.745665,0.750000,346
2,class_weight = balanced,0.50,443,106,80,262,0.791246,0.766082,0.711957,0.738028,368
3,class_weight = balanced,0.45,423,126,69,273,0.781145,0.798246,0.684211,0.736842,399


### 결론

| 모델 | Threshold | Accuracy | Recall | Precision | F1 |
|---|---:|---:|---:|---:|---:|
| baseline | 0.50 | **0.813** | 0.708 | **0.783** | 0.743 |
| baseline | 0.45 | 0.807 | 0.754 | 0.746 | **0.750** |
| balanced | 0.50 | 0.791 | 0.766 | 0.712 | 0.738 |
| balanced | 0.45 | 0.781 | **0.798** | 0.684 | 0.737 |

`balanced + 0.45`는 생존 Recall을 0.798까지 높였지만, Precision과 Accuracy가 크게 낮아졌고 F1도 개선되지 않았다. 두 조정이 모두 생존 예측을 늘리는 방향으로 중복 작용했기 때문이다.

현재 목적이 "생존자를 최대한 놓치지 않는 것"으로 명확하게 정해진 것이 아니라면 **baseline Logistic Regression + threshold 0.45**를 선택하는 것이 가장 균형적이다. Accuracy 감소는 작고, Recall은 높아졌으며, 비교한 후보 중 F1이 가장 높다.

## 11. 다음 해볼 일

1. **Feature의 추가 가치를 다시 비교하기**  
   현재 선택한 `Sex`, `Pclass`, `FamilySize`, `AgeGroup`을 기준으로 `Fare`, `Title`, `Ticket` 관련 feature를 한 번에 하나씩 추가한다. Cabin 분석은 충분히 진행했으므로 우선순위를 낮춘다. 모든 실험은 같은 5-fold CV와 같은 지표로 비교한다.

2. **Threshold와 feature 선택의 과적합 주의하기**  
   같은 OOF 결과에서 후보를 많이 비교하고 가장 좋은 값만 고르면 그 결과에 맞춰질 수 있다. 후보를 정한 뒤 반복 교차검증이나 별도의 validation split으로 선택이 안정적인지 확인한다.

3. **비선형 모델과 비교하기**  
   Logistic Regression 결과를 해석 가능한 기준점으로 남기고, Random Forest·Extra Trees·Gradient Boosting 계열 모델이 feature 간 비선형 관계를 더 잘 포착하는지 비교한다.

4. **Kaggle 제출 파일 만들기**  
   최종 선택안을 확정하면 전체 train 데이터로 Pipeline을 다시 학습하고 test 생존확률에 선택한 threshold를 적용해 `submission.csv`를 만든다. Public score뿐 아니라 사용한 feature, 검증 방식, threshold 선택 이유를 GitHub에 함께 기록한다.